# cositos — live widget in a Jupyter kernel

This notebook builds a **cositos** widget and displays it over the real Jupyter
comm channel. cositos reuses anywidget's `AnyModel`/`AnyView` frontend, so the
widget renders with **no new JavaScript**.

**Prerequisites** (a live kernel is required — a static `.ipynb` will not render
widgets until the `serialize-widget-state` embed feature lands):

- `cositos` importable in the kernel
- `anywidget` installed in the *frontend* (JupyterLab/Notebook) so the
  `anywidget` model/view module resolves — cositos emits `_model_module="anywidget"`.


In [ ]:
# An anywidget-style ESM widget: a counter button. Identical to what you would
# write for anywidget itself.
ESM = """
export default {
  render({ model, el }) {
    const btn = document.createElement("button");
    const out = document.createElement("span");
    const paint = () => { out.textContent = " count = " + model.get("count"); };
    btn.textContent = "increment";
    btn.addEventListener("click", () => {
      model.set("count", model.get("count") + 1);
      model.save_changes();
    });
    model.on("change:count", paint);
    paint();
    el.appendChild(btn); el.appendChild(out);
  }
}
"""

In [ ]:
from cositos import Widget
from cositos.jupyter import CommTransport

# The host owns the state; cositos is the protocol glue (no observer magic).
state = {"_esm": ESM, "count": 0}
transport = CommTransport()
widget = Widget(
    transport,
    get_state=lambda: dict(state),
    set_state=state.update,   # inbound updates from the frontend land here
)
widget.open()   # sends comm_open over the kernel's iopub channel

`Widget` currently exposes `mimebundle()` rather than `_repr_mimebundle_`, so we
display it explicitly. (Ticket: a `_repr_mimebundle_` display shim will let you
just write `widget` as the last line of a cell.)

In [ ]:
from IPython.display import display
display(widget.mimebundle(repr_text='cositos counter'), raw=True)

Click **increment** in the rendered widget: the frontend calls `save_changes()`,
which sends an `update` back over the comm; `set_state` merges it into `state`.
Read it back from the kernel:

In [ ]:
state['count']  # reflects clicks made in the frontend